In [1]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from utils import write_csv
from utils import display_excel_sheets

from models import (
    AssessmentReport,
    DataSource,
    Sector,
    GasType,
    Emissions,
    EmissionsCO2e,
    EmissionsTotalCO2e,
    EmissionsTotalSector,
    AggregationType,
)

def parent_from_code(code):
    if isinstance(code, str) and '.' in code:
        return code.rsplit('.', 1)[0]
    return code


In [ ]:
# carbon monitor cities

datasource_name = "carbon-monitor-cities"
publisher = "carbon-monitor"

# load data and define output directory
data_dir = (
    Path(__file__).resolve().parent.parent.parent / "data" / "carbon-monitor-cities" / "processed"
)
output_dir = data_dir / "processed"
output_dir.mkdir(parents=True, exist_ok=True)



fl = (Path().cwd().parent.parent 
      / "data" 
      / "carbon-monitor-cities" 
      / "raw" 
      / "carbonmonitor-cities_datas_2025-12-05.csv"
      )


#fl = "/Users/lukegloege/projects/ghg-tracker-database/data/carbon-monitor-cities/raw/carbon-monitor-CITIES-maingraphdatas.xlsx"
# carbonmonitor-cities_datas_2025-12-05.csv
#df = pd.read_excel(fl).dropna()

# ------------------------------------------
# Create Locode table
# Note: this may need to be updated with each release
# ------------------------------------------
d = {
    'Qingdao': "CNQIN", 
    'Copenhagen': "DKCPH",  # kobenhavn
    'Paris': "FRPAR", 
    'Berlin': "DEBER", 
    'New Delhi': "INICD", 
    'Tokyo': "JPTYO",
    'Amsterdam': "NLAMS", 
    'Moscow': "RUMOW", 
    'Seoul': "KRSEL",
    'Stockholm': "SESTO",
    'Bangkok': "THBKK",
    'Los Angeles' :"USLAX",
    'Melbourne': "AUMEL",
    'Sydney' : "AUSYD", 
    'Santiago' : "BRSTG",
    'Beijing' : "CNBJS", 
    'Chengdu': "CNCDU",
    'Chongqing' : "CNCQI", 
    'Hangzhou' : "CNHAZ", 
    'Jinan' : "CNJNA", 
    'Nanjing' : "CNNJI", 
    'Shanghai' : "CNSGH",
    'Bogota' : "COBOG", 
    'Helsinki' : "FIHEL", 
    'Lyon': "FRLIO", 
    'Marseille' : "FRMRS", 
    'Nice' : "FRNCE",
    'Munich' : "DEMUC", #Munchen
    'Milan' :"ITMIL", 
    'Rome' :"ITROM", 
    'Osaka':"JPOSA",
    'Mexico City' : "MXMEX", 
    'Oslo' : "NOOSL", 
    'Cape Town' : "ZACPT",
    'Barcelona' : "BRBAQ", 
    'Madrid' : "COMDD", 
    'Istanbul':"TRIST",
    'London' : "GBLON", 
    'Chicago' : "USCHI", 
    'Houston' : "USHOU",
    'Miami' : "USMIA",
    'New York' : "USNYC",
    'San Francisco' : "USSFO", 
    'Seattle' : "USSEA", 
    'Washington': "USWAS"
}

df_locodes = pd.DataFrame(list(d.items()), columns=["city", "locode"])

# ------------------------------------------
# DataSource table
# ------------------------------------------
datasource = DataSource(
    id=f"{publisher}:{datasource_name}",
    name="Carbon Monitor Cities",
    publisher=publisher,
    published_date=datetime.strptime("2015-03-05", "%Y-%m-%d"),
    url="https://cities.carbonmonitor.org/",
)

'''
write_csv(
    output_dir=output_dir,
    name=DataSource.__tablename__,
    data=datasource,
    mode="w",
)
'''

# ------------------------------------------
# Sector table
# ------------------------------------------
sectors_dict = {
    'Aviation': "1", 
    'Ground Transport': "2", 
    'Industry': "3", 
    'Power': "4", 
    'Residential': "5"
}

sector_validated = [
    Sector(
        id=f"carbon-monitor-cities:{code}",
        code=code,
        parent_code=None,
        name=name,
        taxonomy="Carbon Monitor Cities",
        datasource_id=datasource.id,
    ).model_dump()
    for name, code in sectors_dict.items()
]

'''
write_csv(
    output_dir=output_dir,
    name=Sector.__tablename__,
    data=sector_validated,
    mode="w",
)
'''

# ------------------------------------------
# Process data
# Note: file is named at top
# ------------------------------------------

kt_to_tonne = 1_000 # tonne per Kt
df = (pd.read_csv(fl)
    .merge(df_locodes, on='city')  
    .assign(
        date=lambda d: pd.to_datetime(d["date"], dayfirst=True),
        year=lambda d: d["date"].dt.year,
        emissions = lambda d: d["value"].apply(lambda row: row * kt_to_tonne),
        units = "tonne*CO2",
        sector_id = lambda d: d['sector'].replace(sectors_dict)
    )
    .loc[:, ["year", "locode", "sector_id", "emissions", "units"]]
    .groupby(["locode", "year","sector_id", "units", ]).sum()
    .reset_index()
    .assign(gas = "CO2")
    .rename(columns={"locode":"actor_id"})
)


# ------------------------------------------
# emissions
# ------------------------------------------
emissions_validated = [
    Emissions(
        id=f"{row.actor_id}:{row.year}:{row.gas}:{row.sector_id}",
        actor_id=row.actor_id,
        gas=GasType(row.gas),
        sector_id=row.sector_id,
        year=row.year,
        emissions=row.emissions,
        units=row.units,
        datasource_id=datasource.id,
    ).model_dump()
    for row in df.itertuples()
]

write_csv(
    output_dir=output_dir,
    name=Emissions.__tablename__,
    data=emissions_validated,
    mode="w",
)




In [41]:
df.head()

,actor_id,year,sector_id,units,emissions
0,AUMEL,2019,1,tonne*CO2,562491.95
1,AUMEL,2019,2,tonne*CO2,4646046.28
2,AUMEL,2019,3,tonne*CO2,9000517.90
3,AUMEL,2019,4,tonne*CO2,1087340.19
4,AUMEL,2019,5,tonne*CO2,3397557.80


In [26]:
df.head()

,locode,year,sector,value
0,AUMEL,2019,Aviation,562.49195
1,AUMEL,2019,Ground Transport,4646.04628
2,AUMEL,2019,Industry,9000.51790
3,AUMEL,2019,Power,1087.34019
4,AUMEL,2019,Residential,3397.55780


In [ ]:



#write_csv(
#    output_dir=output_dir,
#    name=DataSource.__tablename__,
#    data=datasource.model_dump(),
#    mode="w",
#)

# ------------------------------------------
# Sector table
# ------------------------------------------
fl = data_dir / "raw" / "crt.xlsx"

df = pd.read_excel(fl, sheet_name="Summary1", header=9).loc[
    lambda x: x["Total national emissions and removals"].str.match(r"^\d", na=False)
]

df[["code", "name"]] = df["Total national emissions and removals"].str.split(
    n=1, expand=True
)
df["code"] = df["code"].str.rstrip(".")
df["parent_code"] = df["code"].str.extract(r"^(.*)\.[^.]+$")[0]
df["name"] = df["name"].str.rstrip()
df["name"] = (
    df["name"].str.replace(r"(?:\s*\([^)]+\))+$", "", regex=True).str.strip()
)

df = df.fillna("").astype({"code": str, "parent_code": str, "name": str})

sector_validated = [
    Sector(
        id=f"crt:{row.code}",
        code=row.code,
        parent_code=row.parent_code,
        name=row.name,
        taxonomy="Carbon Monitor Cities",
        datasource_id=datasource.id,
    ).model_dump()
    for row in df.itertuples()
]

write_csv(
    output_dir=output_dir,
    name=Sector.__tablename__,
    data=sector_validated,
    mode="w",
)